# 03. GARDEN and MLP-A2C Baseline Training & Evaluation

This notebook trains and evaluates memory-less neural baselines (CTDE A2C):
1. `GARDENAgent` (Pisacane et al., LoG 2024 — 3-layer GATv2 + node degree).
2. `MLPA2CAgent` (with and without node degree).
3. Learning curve analysis, evaluation of best checkpoints on `test_pairs`, and inspection of loops/failures.

In [1]:
import os
import pickle
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch as T

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.data.splits import load_splits_and_pairs
from src.env.graph_env import GraphNavEnv
from src.models.garden import GARDENAgent, MLPA2CAgent
from src.training.a2c_trainer import A2CTrainer
from src.utils.evaluator import evaluate_agent
from src.utils.metrics import compute_win_rates

sns.set_theme(style="whitegrid")
DEVICE = "cuda" if T.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Experiment Configuration

In [6]:
GRAPH_NAME = "facebook_1"
SEED = 42
NUM_EPISODES = 1000
EVAL_EVERY = 500
VAL_SUBSET_SIZE = 100
TEST_SUBSET_SIZE = 100  # set to None to evaluate on all test_pairs

random.seed(SEED)
np.random.seed(SEED)
T.manual_seed(SEED)

graph_dir = Path("data/processed") / GRAPH_NAME
with open(graph_dir / "graph.pkl", "rb") as f:
    g = pickle.load(f)
features = np.load(graph_dir / "features.npy")
splits, val_pairs, test_pairs = load_splits_and_pairs(str(graph_dir))

train_env = GraphNavEnv(g, features, max_steps=100, target_nodes=splits["train"])
eval_env = GraphNavEnv(g, features, max_steps=100)
print(f"Loaded {GRAPH_NAME}: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")

Loaded facebook_1: 148 nodes, 1692 edges


## 2. Training `GARDENAgent` and `MLPA2CAgent`

In [ ]:
agents = {
    "GARDEN": GARDENAgent(
        env=train_env,
        hidden_dim=64,
        rep_dim=32,
        lr=1e-4,
        entropy_coef=1e-3,
        use_degree=True,
        device=DEVICE,
        name=f"GARDEN_{GRAPH_NAME}",
    ),
    "MLPA2C_WD": MLPA2CAgent(
        env=train_env,
        hidden_dim=64,
        lr=1e-4,
        entropy_coef=1e-3,
        use_degree=True,
        device=DEVICE,
        name=f"MLPA2CWD_{GRAPH_NAME}",
    ),
    "MLPA2C": MLPA2CAgent(
        env=train_env,
        hidden_dim=64,
        lr=1e-4,
        entropy_coef=1e-3,
        use_degree=False,
        device=DEVICE,
        name=f"MLPA2C_{GRAPH_NAME}",
    ),
}

histories = {}
for label, agent in agents.items():
    trainer = A2CTrainer(
        agent=agent,
        env=train_env,
        train_targets=splits["train"],
        val_pairs=val_pairs,
        checkpoint_dir=f"checkpoints/{GRAPH_NAME}",
        eval_every=EVAL_EVERY,
    )
    histories[label] = trainer.train(
        num_episodes=NUM_EPISODES,
        val_subset_size=VAL_SUBSET_SIZE,
        verbose=True,
    )

[A2CTrainer] Starting training for 1000 episodes on GARDEN_facebook_1...
  Ep    500/1000 | Loss: -0.0181 | Val R_oracle: 39.45 | Val R_trunc: 91.0% | Time: 211.4s
    --> New best model saved to checkpoints\facebook_1\GARDEN_facebook_1_best.pt (R_oracle = 39.45)
  Ep   1000/1000 | Loss: -0.0313 | Val R_oracle: 41.45 | Val R_trunc: 96.0% | Time: 398.9s
[A2CTrainer] Starting training for 1000 episodes on MLPA2CWD_facebook_1...
  Ep    500/1000 | Loss: -0.0118 | Val R_oracle: 41.94 | Val R_trunc: 97.0% | Time: 183.4s
    --> New best model saved to checkpoints\facebook_1\MLPA2CWD_facebook_1_best.pt (R_oracle = 41.94)


## 3. Validation Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, hist in histories.items():
    steps_x = [EVAL_EVERY * (i + 1) for i in range(len(hist["val_oracle_ratio"]))]
    axes[0].plot(steps_x, hist["val_oracle_ratio"], marker="o", label=label)
    axes[1].plot(steps_x, hist["val_trunc"], marker="o", label=label)

axes[0].set_title(f"Validation R_bar_oracle ({GRAPH_NAME})")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("R_bar_oracle")
axes[0].legend()

axes[1].set_title(f"Validation R_trunc % ({GRAPH_NAME})")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("R_trunc (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Test Set Evaluation of Best Checkpoints & Loop Inspection

In [ ]:
test_sub = test_pairs[:TEST_SUBSET_SIZE] if TEST_SUBSET_SIZE else test_pairs
test_summaries = []
test_records = {}
path_lengths = {}

for label, agent in agents.items():
    ckpt_path = Path(f"checkpoints/{GRAPH_NAME}/{agent.name}_best.pt")
    if ckpt_path.exists():
        agent.load_checkpoint(str(ckpt_path))
    agent.eval()
    summary, recs = evaluate_agent(agent, eval_env, test_sub, show_progress=False)
    summary["model"] = label
    test_summaries.append(summary)
    test_records[label] = recs
    path_lengths[label] = [r["steps"] for r in recs]

win_rates = compute_win_rates(path_lengths, seed=SEED)
for s in test_summaries:
    s["win_rate"] = win_rates[s["model"]]

df_test = pd.DataFrame(test_summaries)
display(df_test[["model", "r_trunc", "mean_oracle_ratio", "oracle_ratio_ci", "mean_revisit_rate", "mean_ping_pong_count", "win_rate"]])

garden_failed = [r for r in test_records["GARDEN"] if r["truncated"]]
print(f"\nGARDEN truncated episodes: {len(garden_failed)}/{len(test_sub)}")
for r in garden_failed[:3]:
    print(f"  src={r['source']} -> tgt={r['target']} | sp={r['shortest_path_len']} | ping_pong={r['ping_pong_count']} | cycles={r['cycle_count']} | traj[:20]={r['trajectory'][:20]}")